# 08 · HEADPATH: train-time cell-head clamp experiment (DIAGNOSTIC DEVIATION; OFF by default)

Plan, decisions and outcomes: `docs/HEADPATH_PLAN.md`. Rationale: `docs/POST_COLLAPSE_FORENSIC_REPORT.md` §11.

> **Not a reproduction.** Arm **U** deviates from the official model: the cell head's `clamp(min=0)` is omitted in
> training forward passes, so the official loss sees raw cell outputs. Arm **O** is the unmodified official model.
> Every evaluation of both arms uses the official clamped forward and the official metrics.

* **Runs.** 12 primary runs (2 arms × 6 independent paired seeds, 42–47) + 2 separate test–retest runs (both arms,
  seed 42; never used for statistical inference) = **14 runs × 490 steps** (2 epochs of the notebook-07 FULL set,
  245 graphs), FP32. Validation: the notebook-07 12-graph / 7-design set, every 49 steps and at step 0.
* **Cell 1 (preflight, CPU only, GPU hidden):** the full test suite, including the 4 real-data tests on a 3-sample
  subset of the raw tree extracted by notebook 02 in this session.
* **Cell 2 (plan, no GPU):** recomputes the plan from the processed data on Drive and refuses to continue if it differs
  from `configs/headpath_manifest.json`, from the notebook-07 manifest, from the notebook-07 plan on Drive, or if the
  labels differ from the notebook-04 report.
* **Cell 3 (run):** off until approved. Content-hashes every protected notebook 05–07 file (smoke and benchmark run
  folders, `runs/collapse_diag/`, the 05–07 reports) before and, always, after, and fails if anything changed. Every
  run is isolated: a failure is recorded (FAILED, with its traceback) and the next run starts; after a runtime-fatal
  error the rest are recorded as NOT_STARTED. The report lists all 14 runs and is rewritten atomically after every
  run. The **last** step always flushes Drive (`flush_and_unmount`) and, by default, remounts it to verify the
  reports. Outputs: `runs/headpath/<run>/` and `reports/08_headpath_*.json`; existing files are never overwritten.
* **Estimated A100 time.** About 55–110 min including graph building (≈ 4–7 min per run), and ≈ 0.7 GB of Drive.
  This is an estimate, not a measurement.

Set `RUN_HEADPATH = True` in the last cell **only after explicit approval**.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
# Preflight (CPU only: CUDA is hidden from the test process, so no GPU work happens here).
import os, subprocess, sys
from pathlib import Path
from verihgn_repro.data import circuitnet as cn
REQUIRE_REAL_DATA_TESTS = True     # the 4 tests in tests/test_real_data.py need the raw tree from notebook 02
env = dict(os.environ, CUDA_VISIBLE_DEVICES='')
if (L.local_raw / 'instance_placement_gcell').exists():
    fx = cn.subset_raw_tree(L.local_raw, Path('/content/real_fixture_tree'))   # symlinks only; nothing copied
    env['VERIHGN_FIXTURES'] = str(fx)
    print('real-data fixture tree:', fx, sorted(p.name for p in (fx / 'instance_placement_gcell').iterdir()))
else:
    print('no raw tree at', L.local_raw, '- run notebook 02 in this session to enable the real-data tests')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-rs', '-p', 'no:cacheprovider', 'tests'],
                   cwd=CODE_DIR, env=env, capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-2000:])
assert r.returncode == 0, 'test suite failed: do not continue'
real_skipped = 'VERIHGN_FIXTURES' in r.stdout
assert not (REQUIRE_REAL_DATA_TESTS and real_skipped), 'the real-data tests were skipped: run notebook 02 first'
print('preflight passed | real-data tests ran:', not real_skipped)

In [ ]:
import json, time
from pathlib import Path
from verihgn_repro.config import load_config
from verihgn_repro.data.splits import load_manifest
from verihgn_repro.diagnostics import headpath as H
SCOPE = 'official9'
CFG_DIR = Path(CODE_DIR) / 'configs'
PHASH = json.load(open(L.reports / f'03_preprocessing_{SCOPE}.json'))['prep_hash']
PROCESSED = L.processed_for(PHASH)
cfg = load_config(CFG_DIR / 'base.yaml')                  # official hyper-parameters; primary variant
H.check_base_config(cfg)
man = load_manifest(L.splits / f'{SCOPE}__{cfg.experiment.split}.json')
EXPECTED = json.load(open(CFG_DIR / 'headpath_manifest.json'))               # pre-registration
C07 = json.load(open(CFG_DIR / 'collapse_experiment_manifest.json'))         # FULL / validation of notebook 07
plan = H.plan_headpath(PROCESSED, man, expected=EXPECTED, collapse_manifest=C07)   # raises if anything differs
val04 = json.load(open(L.reports / f'04_validation_{SCOPE}.json'))
assert abs(plan['c_max'] - val04['c_max'][cfg.experiment.split]) < 1e-9, 'labels differ from the validated ones'
p07 = L.reports / '07_collapse_plan.json'                 # the plan notebook 07 actually ran with (read only)
if p07.exists():
    p07 = json.load(open(p07))
    assert p07['arms']['FULL'] == plan['train'] and p07['val'] == plan['val'], 'differs from the executed 07 plan'
    assert abs(p07['c_max'] - plan['c_max']) < 1e-9, 'c_max differs from the executed 07 plan'
    print('matches the executed notebook-07 plan (FULL, validation set, c_max)')
missing = [n for n, d in plan['sample_dirs'].items() if not (Path(d) / 'sample.npz').exists()]
assert not missing, f'processed samples missing on Drive: {missing}'
primary = [r for r in plan['runs'] if not r['retest']]
retest = [r for r in plan['runs'] if r['retest']]
assert (len(primary), len(retest), len(plan['runs'])) == (12, 2, 14)
assert sorted({r['seed'] for r in primary}) == list(range(42, 48)) and [(r['arm'], r['seed']) for r in retest] == [('O', 42), ('U', 42)]
assert not plan['pending_research_decisions'], plan['pending_research_decisions']

def new_report(name):                     # never an existing file (timestamp + counter if needed)
    return H.new_report_path(L.reports, name)

plan_path = new_report('08_headpath_plan.json')
plan_path.write_text(json.dumps({k: v for k, v in plan.items() if k != 'sample_dirs'}, indent=1))
print('c_max', plan['c_max'], '| train', len(plan['train']), '| val', len(plan['val']),
      '| steps per run', plan['steps_per_run'], '| evaluations at', plan['eval_steps'])
for r in plan['runs']:
    print(f"{r['name']:28s} arm {r['arm']}  seed {r['seed']}  {'RETEST (excluded from inference)' if r['retest'] else 'primary'}"
          f"  cell_head_train_clamp={r['cell_head_train_clamp']}  steps {r['steps']}")
print('plan written to', plan_path)

In [ ]:
RUN_HEADPATH = False      # set True only after explicit approval
VERIFY_AFTER_REMOUNT = True   # after flush_and_unmount, remount Drive and re-hash the reports (may show the auth popup)
if RUN_HEADPATH:
    from verihgn_repro import colab
    from verihgn_repro.data.graph import build_graph
    from verihgn_repro.data.preprocess import load_sample

    def build_graphs():
        t0 = time.time()
        graphs = {}
        for name, d in plan['sample_dirs'].items():
            a, m = load_sample(Path(d))
            graphs[name] = build_graph(a, m, grid_features=cfg.experiment.grid_features,
                                       label_norm=cfg.experiment.label_norm, c_max=plan['c_max'])
        print(f'{len(graphs)} graphs built in {time.time() - t0:.0f}s')
        return graphs

    report_path = new_report('08_headpath_experiment.json')
    # execute(): protected-artifact hashes -> graphs -> every run isolated, report after every run -> (always)
    # re-hash + record -> (always, last) Drive flush_and_unmount + verification. Raises at the end on any failure.
    out = H.execute(plan, cfg, L, report_path, build_graphs, flush=lambda paths: colab.flush_drive(paths, remount_verify=VERIFY_AFTER_REMOUNT), verify_paths=[plan_path])
    rep = out['report']
    print(json.dumps({k: rep['analysis'][k] for k in ('verdict', 'run_statuses', 'category_counts', 'primary_test', 'hsig',
                                                      'historical_replication', 'conclusion')}, indent=1, default=str))
    if rep['analysis']['experiment_status'] != 'VALID':
        print('\n' + '!' * 100 + f"\nEXPERIMENT STATUS {rep['analysis']['experiment_status']}: "
              f"{rep['analysis']['status_reasons']}\nFailed runs carry their traceback in the report; re-run this cell to "
              "resume (completed runs are not retrained).\n" + '!' * 100)
    print('Drive was flushed' + (' and remounted' if VERIFY_AFTER_REMOUNT else ' and UNMOUNTED (re-run the bootstrap cell to continue)'))
    print('→ send back', report_path.name, 'and', plan_path.name)
else:
    print('not run (RUN_HEADPATH = False)')